# Automated Lightning AI Training Notebook: Hana Kim (EA-F-001) Flux.1-dev LoRA
This notebook automates the entire training setup in your new Lightning AI Studio. It clones the repository, installs dependencies, handles conflicts, extracts the dataset, writes the configuration, and launches the training process.

### Step 1: Clone Repository and Install Requirements
Run this cell to clone `ai-toolkit` and install all required python libraries.

In [ ]:
# Remove any old folder and clone a fresh copy of the trainer repo
!rm -rf /teamspace/studios/this_studio/ai-toolkit
!git clone --recursive https://github.com/ostris/ai-toolkit.git /teamspace/studios/this_studio/ai-toolkit

# Install requirements
%cd /teamspace/studios/this_studio/ai-toolkit
!pip install -r requirements.txt
!pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu121

### Step 2: Auto-Patch System torchaudio Mismatch
This cell creates a local dummy `torchaudio` package inside `ai-toolkit` which overrides the broken system library, ensuring zero load or symbol errors.

In [ ]:
import os

# Create local dummy package
dummy_dir = "/teamspace/studios/this_studio/ai-toolkit/torchaudio"
os.makedirs(dummy_dir, exist_ok=True)
with open(os.path.join(dummy_dir, "__init__.py"), "w") as f:
    f.write("# Dummy package to override system torchaudio conflicts\n")

print("Successfully created local torchaudio override in ai-toolkit/torchaudio!")

### Step 3: Extract Uploaded Dataset
This cell unzips your `character_id.zip` dataset file to the correct teamspace path.

In [ ]:
import zipfile
import os

zip_path = "/teamspace/studios/this_studio/MODEL/character_id.zip"
dest_path = "/teamspace/studios/this_studio/"

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dest_path)
    print("Extraction complete!")
else:
    print(f"Warning: Could not find {zip_path}. Please upload character_id.zip to /teamspace/studios/this_studio/MODEL/ before continuing.")

### Step 4: Write Training Configuration
This cell generates the YAML configuration file required by `ai-toolkit` for training the LoRA.

In [ ]:
yaml_config = """
job: extension
config:
  name: "ea_f_001_hana_kim_lora"
  process:
    - type: 'sd_trainer'
      training_folder: "/teamspace/studios/this_studio/output"
      performance_log_every: 250
      device: cuda
      trigger_word: "ea_f_001_hana_kim"
      network:
        type: "lora"
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 500
        max_step_saves_to_keep: 4
      datasets:
        - folder_path: "/teamspace/studios/this_studio/character_id/train"
          caption_ext: "txt"
          resolution: [512, 768, 1024]
      train:
        epochs: 45
        steps: 2500
        gradient_accumulation_steps: 1
        learning_rate: 1e-4
        batch_size: 1
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: "adamw8bit"
        ema_decay: 0.99
        dtype: bf16
      model:
        name_or_path: "black-forest-labs/FLUX.1-dev"
        is_flux: true
        quantize: true
      sample:
        sampler: "flowmatch"
        sample_every: 250
        width: 1024
        height: 1024
        prompts:
          - "ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, wearing elegant red halterneck dress, outdoor garden background, high fashion editorial, 85mm portrait"
          - "ea_f_001_hana_kim, Hana Kim, 28-year-old East Asian Korean woman, wearing cream sleeveless knit dress, luxury studio background, direct gaze, soft lighting"
"""

config_dir = "/teamspace/studios/this_studio/ai-toolkit/config"
os.makedirs(config_dir, exist_ok=True)
with open(os.path.join(config_dir, "flux_training_config.yaml"), "w") as f:
    f.write(yaml_config)
print("Training configuration written to /teamspace/studios/this_studio/ai-toolkit/config/flux_training_config.yaml")

### Step 5: Start Training Job
Run this cell to start the Flux.1-dev LoRA training. It will update the step logs in real time.

In [ ]:
%cd /teamspace/studios/this_studio/ai-toolkit
!python run.py config/flux_training_config.yaml

### Step 6: Export Model Weights
Your trained model file `ea_f_001_hana_kim_lora.safetensors` will be generated inside `/teamspace/studios/this_studio/output/ea_f_001_hana_kim_lora`. You can download it directly from the left sidebar file manager.